# Bayesian linear regression

_Classical ML_

**Don't fit one line — keep a whole distribution of plausible lines.**

Ordinary linear regression returns one best line. But with little data, many lines fit almost as well.

     Bayesian linear regression keeps all of them, weighted by how plausible each is.

---

This notebook is a practice scaffold for the **Bayesian linear regression** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import BayesianRidge

X, y, coef = make_regression(n_samples=120, n_features=4, noise=8.0,
                             coef=True, random_state=0)

br = BayesianRidge(compute_score=True)
br.fit(X, y)

print("true coefficients :", np.round(coef, 2))
print("learned coef (mean):", np.round(br.coef_, 2))
print("intercept:", round(br.intercept_, 3))
print("alpha (noise precision):", round(br.alpha_, 4))
print("lambda (weight precision):", round(br.lambda_, 4))

mean, std = br.predict(X[:3], return_std=True)
for i in range(3):
    print("pred %.2f +/- %.2f   (true %.2f)" % (mean[i], std[i], y[i]))

## Visualize the data & results

_Fitting diabetes progression from BMI, how does the Bayesian posterior slope compare to plain OLS?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import BayesianRidge, LinearRegression

dia = load_diabetes()
x = dia.data[:, 2]            # BMI feature
y = dia.target.astype(float)
X = x.reshape(-1, 1)

ols = LinearRegression().fit(X, y)
br = BayesianRidge().fit(X, y)
print(ols.coef_[0], br.coef_[0])

xs = np.linspace(x.min() - 0.01, x.max() + 0.01, 30).reshape(-1, 1)
plt.scatter(x, y, c="#ffb454", s=30)
plt.plot(xs.ravel(), ols.predict(xs), c="#ff7b72", linestyle="--",
         label="OLS fit")
plt.plot(xs.ravel(), br.predict(xs), c="#4ea1ff",
         label="posterior mean")
plt.plot(xs.ravel(), np.full_like(xs.ravel(), y.mean()), c="#9aa7b4",
         linestyle="--", label="prior mean")
plt.legend()
plt.title("Bayesian ridge vs OLS on Diabetes (BMI)")
plt.xlabel("BMI (mean-centered, scaled)")
plt.ylabel("disease progression")
plt.show()